# B7 · Du notebook à la production — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b7_production/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B7.Q1 à B7.Q11, B7.R1 à B7.R3, B7.1 à B7.4 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B7.9 à B7.15 | le projet : du notebook au paquet, ligne de commande, configuration, reproductibilité, suivi des runs et tests (*prochaine session de génération*) | | | |
| B | B7.16 à B7.21 | le service : les écarts entre entraînement et production, l'artefact, l'API, la latence, l'accélération sur CPU et l'export (*prochaine session de génération*) | | | |
| C | B7.22 à B7.25 | l'exploitation : la dérive, l'intégration continue, le Dockerfile et le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Restructurer un notebook en paquet Python installable, avec une ligne de commande, du logging et une configuration validée.
- Rendre un entraînement reproductible et traçable (graines, déterminisme, versions, empreinte des données, journal des runs), et rapporter un score sur plusieurs graines avec son intervalle.
- Écrire des tests utiles à un projet de ML et les automatiser dans une intégration continue.
- Empaqueter un modèle avec son prétraitement, le servir par une API HTTP et mesurer sa latence.
- Exporter et accélérer l'inférence sur CPU en vérifiant l'équivalence numérique ; détecter une dérive des données et décider d'une action.

**Rappel express.** Intervalle sur $n$ graines : $\bar a \pm t_{0{,}975;\,n-1}\,\frac{s}{\sqrt n}$ ($s$ divisé par $n - 1$). Percentile de `np.percentile` : position $h = \frac{q}{100}(n - 1)$ dans les mesures triées, interpolation linéaire. Débit $= \frac{\text{taille du batch}}{\text{latence du batch}}$ ; loi de Little $L = \lambda W$. Taille des poids $= N \times$ octets par paramètre (4, 2, 1). $\operatorname{PSI} = \sum_k (p_k - q_k)\ln\frac{p_k}{q_k} = D_{\text{KL}}(p\,\|\,q) + D_{\text{KL}}(q\,\|\,p)$ ; $D = \sup_x |F_{\text{réf}}(x) - F_{\text{prod}}(x)|$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B7.1 à B7.4)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres ensemble, entre guillemets, dans n'importe quel ordre (`"EF"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », l'estimation 🧮 B7.5, la réflexion (🗣️ B7.6, ⚖️ B7.7, 📄 B7.8) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np

MB, MIB = 1e6, 2 ** 20                  # 1 Mo = 10**6 bytes, 1 Mio = 2**20 bytes (fiche, 207.8)

# quiz Q1: the cells x = 2 ; x = x * 10 ; print(x + 1), run in the order 1, 2, 3, 2, 3
Q1A = 2 * 10 * 10 + 1
# quiz Q3: basicConfig(level=WARNING), then 1 debug, 3 info, 1 warning and 1 error messages
LEVELS_Q3 = ["DEBUG"] + ["INFO"] * 3 + ["WARNING", "ERROR"]
Q3C = sum(level in ("WARNING", "ERROR", "CRITICAL") for level in LEVELS_Q3)
# quiz Q7: the ML Test Score is the minimum of the four section sums
SECTIONS_Q7 = {"data": 3.0, "model": 2.5, "infrastructure": 4.0, "monitoring": 1.0}
Q7E = min(SECTIONS_Q7.values())
# quiz Q9: 900 requests of 10 ms and 100 of 200 ms ; Little's law ; a throughput
LAT_Q9 = np.array([10.0] * 900 + [200.0] * 100)
Q9A, Q9B = float(LAT_Q9.mean()), float(np.percentile(LAT_Q9, 95))
Q9D = 60 * 0.150                         # L = lambda W, W in seconds
Q9E = 64 / 0.4
# recall R1: a JSONL log of 7 runs, 2 runs added (append, never rewrite)
R1D = 7 + 2

# ✏️ B7.1: ConvNeXt-Tiny (torchvision) and the VGG of the MP5 solution
N_B71, LIN_W_B71, LIN_B_B71 = 28_589_128, 26_646_528, 34_120
F32_B71 = 4 * N_B71 / MB
F32_MIB_B71 = 4 * N_B71 / MIB
BF16_B71 = 2 * N_B71 / MB
INT8_B71 = N_B71 / MB
DYNQ_B71 = (LIN_W_B71 + 4 * (N_B71 - LIN_W_B71)) / MB          # dense matrices in int8, everything else float32
RATIO_B71 = F32_B71 / DYNQ_B71
PEAK_B71 = 2 * F32_MIB_B71                                     # random weights + the copy read by torch.load
VGG_PARAMS_B71, VGG_LINEAR_B71, VGG_FILE_B71 = 72_954, 650, 296_464
VGG_BN_CHANNELS_B71 = (16, 16, 32, 32, 64, 64)
VGG_FLOAT_BUFFERS_B71 = 2 * sum(VGG_BN_CHANNELS_B71)            # running_mean and running_var
VGG_BYTES_B71 = 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71) + 8 * len(VGG_BN_CHANNELS_B71)
HEADER_B71 = VGG_FILE_B71 - VGG_BYTES_B71
LIN_SHARE_B71 = 100 * VGG_LINEAR_B71 / VGG_PARAMS_B71

# ✏️ B7.2: five seeds, Student t with n - 1 = 4 degrees of freedom (the table of 207.4)
A_B72 = np.array([64.2, 65.0, 63.6, 64.8, 64.4])
B_B72 = np.array([64.9, 65.3, 64.5, 65.6, 64.7])
T4_B72 = 2.776


def interval_b72(values, t=T4_B72, ddof=1):
    """Mean, standard deviation, standard error and half-width of the Student interval."""
    m, s = float(np.mean(values)), float(np.std(values, ddof=ddof))
    se = s / math.sqrt(len(values))
    return m, s, se, t * se


MEAN_A_B72, S_A_B72, SE_A_B72, HW_A_B72 = interval_b72(A_B72)
LOW_A_B72 = MEAN_A_B72 - HW_A_B72
MEAN_B_B72, _, _, HW_B_B72 = interval_b72(B_B72)
OVERLAP_B72 = bool(MEAN_A_B72 + HW_A_B72 >= MEAN_B_B72 - HW_B_B72)
MEAN_D_B72, S_D_B72, SE_D_B72, HW_D_B72 = interval_b72(B_B72 - A_B72)
ZERO_IN_D_B72 = bool(MEAN_D_B72 - HW_D_B72 <= 0 <= MEAN_D_B72 + HW_D_B72)

# ✏️ B7.3: twenty latencies of batch 1 (ms), np.percentile's default (method="linear")
LAT_B73 = np.array([21.3, 19.8, 20.4, 22.1, 19.6, 20.9, 20.1, 35.7, 20.6, 19.9,
                    21.0, 20.2, 23.4, 19.7, 20.5, 47.9, 20.8, 20.3, 21.6, 20.0])
MEAN_B73 = float(LAT_B73.mean())
P50_B73, P95_B73 = (float(v) for v in np.percentile(LAT_B73, [50, 95]))
ABOVE_B73 = int((LAT_B73 > MEAN_B73).sum())
P50_32_B73, P95_32_B73 = 118.0, 131.0                          # batch 32, given by the statement
THR1_B73 = 1000 / P50_B73
THR32_B73 = 32 * 1000 / P50_32_B73
RATIO_B73 = THR32_B73 / THR1_B73
PER_IMAGE_B73 = P50_32_B73 / 32
SLO_B73 = bool(P95_32_B73 <= 100)
PAGE_B73 = 1 - 0.95 ** 8

# ✏️ B7.4: PSI on four intervals (the quartiles of the reference), natural log, epsilon added to every share
Q_B74 = np.full(4, 0.25)
COUNTS_B74 = {1: [38, 52, 54, 56], 2: [96, 62, 30, 12], 3: [120, 56, 24, 0]}
P_B74 = {week: np.array(c) / 200 for week, c in COUNTS_B74.items()}


def psi_terms_b74(p, q=Q_B74, eps=0.0, log=np.log):
    """The four terms (p_k - q_k) log(p_k / q_k), eps added to every share of p and q (without renormalizing)."""
    p, q = np.asarray(p, float) + eps, np.asarray(q, float) + eps
    return (p - q) * log(p / q)


C1_W1_B74 = float(psi_terms_b74(P_B74[1])[0])
PSI_W1_B74 = float(psi_terms_b74(P_B74[1]).sum())
PSI_W2_B74 = float(psi_terms_b74(P_B74[2]).sum())
TOP_W2_B74 = "ABCD"[int(np.argmax(psi_terms_b74(P_B74[2])))]
KL_PQ_W2_B74 = float(np.sum(P_B74[2] * np.log(P_B74[2] / Q_B74)))
KL_QP_W2_B74 = float(np.sum(Q_B74 * np.log(Q_B74 / P_B74[2])))
PSI_W3_B74 = float(psi_terms_b74(P_B74[3], eps=1e-4).sum())
PSI_W3_EPS3_B74 = float(psi_terms_b74(P_B74[3], eps=1e-3).sum())

# The same answers from the libraries (no download): NumPy, SciPy, torchvision on the `meta` device, safetensors,
# PyYAML, logging, argparse, subprocess, DataLoader, torch.export, torch.load and scikit-learn
import io
import json
import logging
import subprocess
import sys
import tempfile
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import yaml
from scipy import stats

# quiz Q1: run the three cells in the order 1, 2, 3, 2, 3 and keep what the last one prints
cells, namespace, printed = {1: "x = 2", 2: "x = x * 10", 3: "shown = x + 1"}, {}, None
for k in (1, 2, 3, 2, 3):
    exec(cells[k], namespace)
assert namespace["shown"] == Q1A

# quiz Q3: a WARNING level lets through WARNING and ERROR only ; an exception swallowed by print ends with code 0
records = []
logger = logging.getLogger("wb_chB7_q3")
logger.propagate = False
handler = logging.Handler(level=logging.WARNING)
handler.emit = records.append
logger.addHandler(handler)
logger.setLevel(logging.DEBUG)
for level in LEVELS_Q3:
    logger.log(getattr(logging, level), "message")
logger.removeHandler(handler)
assert len(records) == Q3C
swallow = "try:\n    raise RuntimeError('epoch 2: loss is nan')\nexcept Exception as err:\n    print(err)\n"
assert subprocess.run([sys.executable, "-c", swallow], capture_output=True).returncode == 0
unknown = "import argparse; p = argparse.ArgumentParser(); p.add_argument('--lr'); p.parse_args(['--lerning-rate', '1'])"
assert subprocess.run([sys.executable, "-c", unknown], capture_output=True).returncode == 2

# quiz Q4: PyYAML 6.0 reads 5.0e-4 as a float, 5e-4 and "5.0e-4" as strings ; a dataclass does not check types
assert yaml.safe_load("lr: 5.0e-4")["lr"] == 0.0005
assert yaml.safe_load("lr: 5e-4")["lr"] == "5e-4" and yaml.safe_load('lr: "5.0e-4"')["lr"] == "5.0e-4"


@dataclass(frozen=True)
class FrozenOnly:
    lr: float = 0.05


assert FrozenOnly(lr="1e-3").lr == "1e-3"                       # accepted: no check of the annotation

# quiz Q5: float addition is not associative ; without a generator, an extra draw moves the DataLoader's order
assert ((0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3)) is False
import torch
from torch.utils.data import DataLoader


def first_order(extra_draw):
    torch.manual_seed(0)
    if extra_draw:
        torch.rand(1)
    return next(iter(DataLoader(list(range(10)), batch_size=10, shuffle=True))).tolist()


assert first_order(False) != first_order(True)

# quiz Q7, Q9 and recall R1
assert Q7E == 1.0 and Q9A == 29.0 and Q9B == 200.0 and math.isclose(Q9D, 9.0) and math.isclose(Q9E, 160.0)
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "runs.jsonl"
    for seed in list(range(7)) + [10, 11]:                     # 7 runs, then 2 runs again with other seeds
        with path.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"seed": seed}) + "\n")
    assert len(path.read_text(encoding="utf-8").splitlines()) == R1D
assert datetime.strptime("20261104-161731", "%Y%m%d-%H%M%S").strftime("%H:%M:%S") == "16:17:31"

# quiz Q8: the classes sorted by name ; safetensors metadata must be strings
classes = ["avion", "automobile", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]
assert classes[3] == "chat" and sorted(classes)[3] == "camion"
from safetensors.torch import save as save_safetensors

try:
    save_safetensors({"w": torch.zeros(2)}, metadata={"version": 1})
except TypeError:
    pass
else:
    raise AssertionError("safetensors accepted a metadata value that is not a string")

# quiz Q10: an export without dynamic_shapes refuses another batch size ; a data-dependent if does not export
small = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(12, 3)).eval()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    exported = torch.export.export(small, (torch.randn(8, 3, 2, 2),)).module()
    try:
        exported(torch.randn(3, 3, 2, 2))
    except Exception as err:                                    # « Guard failed: input.size()[0] == 8 »
        assert "8" in str(err)
    else:
        raise AssertionError("the exported model accepted a batch of 3")


class DataDependent(torch.nn.Module):
    def forward(self, x):
        if x.mean() > 0:
            return 2 * x
        return x - 1


logging.getLogger("torch").setLevel(logging.CRITICAL)          # torch.export logs the failed trace
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.export.export(DataDependent(), (torch.randn(4, 3),))
except Exception as err:
    assert "data-dependent" in str(err), str(err)[:200]
else:
    raise AssertionError("a data-dependent if was exported")

# recall R2: weights_only=True refuses an arbitrary Python object


class Trainer:
    pass


buffer = io.BytesIO()
torch.save({"trainer": Trainer()}, buffer)
buffer.seek(0)
try:
    torch.load(buffer, weights_only=True)
except Exception:
    pass
else:
    raise AssertionError("weights_only=True loaded a Python object")

# recall R3: the warning of scikit-learn when a model was written by another version
from sklearn.exceptions import InconsistentVersionWarning

assert issubclass(InconsistentVersionWarning, UserWarning)

# ✏️ B7.1: ConvNeXt-Tiny counted on the `meta` device (no memory, no download) ; the MP5 VGG and its safetensors
from torchvision.models import convnext_tiny

with torch.device("meta"):
    convnext = convnext_tiny(weights=None)
linear = [m for m in convnext.modules() if isinstance(m, torch.nn.Linear)]
assert sum(p.numel() for p in convnext.parameters()) == N_B71 and not list(convnext.buffers())
assert sum(m.weight.numel() for m in linear) == LIN_W_B71 and sum(m.bias.numel() for m in linear) == LIN_B_B71


def vgg_mp5(widths=(16, 32, 64)):
    """The VGG of the MP5 solution (cifarkit.models.vgg with downsample="pool" and batchnorm)."""
    layers, c = [], 3
    for i, w in enumerate(widths):
        for _ in range(2):
            layers += [torch.nn.Conv2d(c, w, 3, padding=1, bias=False), torch.nn.BatchNorm2d(w), torch.nn.ReLU()]
            c = w
        if i < len(widths) - 1:
            layers.append(torch.nn.MaxPool2d(2))
    layers += [torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten(), torch.nn.Linear(c, 10)]
    return torch.nn.Sequential(*layers)


vgg = vgg_mp5()
state = vgg.state_dict()
assert sum(p.numel() for p in vgg.parameters()) == VGG_PARAMS_B71
assert sum(t.numel() * t.element_size() for t in state.values()) == VGG_BYTES_B71
assert len(save_safetensors(state)) == VGG_FILE_B71
assert sum(p.numel() for p in vgg[-1].parameters()) == VGG_LINEAR_B71

# ✏️ B7.2: the Student intervals of SciPy (exact quantile 2.7764 instead of the 2.776 of the table)
assert math.isclose(stats.t.ppf(0.975, 4), T4_B72, abs_tol=5e-4)
ci_a = stats.ttest_1samp(A_B72, 0).confidence_interval()
ci_d = stats.ttest_rel(B_B72, A_B72).confidence_interval()
assert round(float(ci_a.low), 2) == round(LOW_A_B72, 2)
assert round(float(ci_d.high - ci_d.low) / 2, 2) == round(HW_D_B72, 2) and (float(ci_d.low) > 0) == (not ZERO_IN_D_B72)

# ✏️ B7.3: the other conventions of np.percentile, for the record
assert np.percentile(LAT_B73, 95, method="inverted_cdf") == 35.7 and np.percentile(LAT_B73, 50, method="inverted_cdf") == 20.5

# ✏️ B7.4: PSI = KL(p || q) + KL(q || p), with scipy.stats.entropy
for week in (1, 2):
    kl = stats.entropy(P_B74[week], Q_B74) + stats.entropy(Q_B74, P_B74[week])
    assert math.isclose(kl, float(psi_terms_b74(P_B74[week]).sum()))
assert math.isclose(stats.entropy(Q_B74, P_B74[2]), KL_QP_W2_B74)
with np.errstate(divide="ignore", invalid="ignore"):
    assert np.isinf(psi_terms_b74(P_B74[3]).sum())
print("✅ the paper answers agree with NumPy, SciPy, torchvision, safetensors, PyYAML, logging, subprocess and torch")




def forward(self, arg0_1: "f32[4, 3]"):
    # File: /tmp/ipykernel_7579/849750965.py:198 in forward, code: if x.mean() > 0:
    mean: "f32[]" = torch.ops.aten.mean.default(arg0_1);  arg0_1 = None
    gt: "b8[]" = torch.ops.aten.gt.Scalar(mean, 0);  mean = None
    ne: "b8[]" = torch.ops.aten.ne.Scalar(gt, 0);  gt = None
    item: "Sym(Eq(u0, 1))" = torch.ops.aten.item.default(ne);  ne = item = None
    



def forward(self, arg0_1: "f32[4, 3]"):
    # File: /tmp/ipykernel_7579/849750965.py:198 in forward, code: if x.mean() > 0:
    mean: "f32[]" = torch.ops.aten.mean.default(arg0_1);  arg0_1 = None
    gt: "b8[]" = torch.ops.aten.gt.Scalar(mean, 0);  mean = None
    ne: "b8[]" = torch.ops.aten.ne.Scalar(gt, 0);  gt = None
    item: "Sym(Eq(u0, 1))" = torch.ops.aten.item.default(ne);  ne = item = None
    


✅ the paper answers agree with NumPy, SciPy, torchvision, safetensors, PyYAML, logging, subprocess and torch


**B7.Q1 — Les pièges du notebook : état caché et ordre d'exécution**

In [3]:
wb.record("B7.Q1a", Q1A, mistakes={"le noyau garde l'ordre où tu as exécuté les cellules, pas celui où elles sont écrites : refais la suite 1, 2, 3, 2, 3": 21,
          "compte les exécutions de la cellule 2 dans la suite 1, 2, 3, 2, 3": 2001,
          "relis la cellule 3 : elle affiche x + 1": 200})
wb.record("B7.Q1b", "A", mistakes={"« Redémarrer » vide le noyau : d'où viendrait encore scale ?": "B", "Jupyter n'exécute que les cellules du fichier : relis « Ce que cache un notebook » (§207.1)": "C"})
wb.record("B7.Q1c", False, mistakes={"un .ipynb range code, sorties et images en JSON : relis « Ce que cache un notebook » (§207.1)": True})
wb.record("B7.Q1d", "C", mistakes={"un module déjà importé est-il relu par le noyau ? Relis la fin de « pyproject.toml et installation éditable »": "A", "le lien éditable existe déjà : est-ce l'installation ou le noyau qui garde l'ancienne version ?": "B"})
wb.record("B7.Q1e", True, mistakes={"supprimer une cellule touche-t-il au noyau ? Relis « Ce que cache un notebook » (§207.1)": False})

WB_ANSWER {"hash": "c60b08e131923f38ef20943d6b7a1b80374e12ff8cc80644acb9d63e54cc97d1", "id": "B7.Q1a", "kind": "int", "magnitude": 2, "mistakes": {"949a78b1c4a3833b78c74333fd5106725777ba5f3d00068d1f08f9ff77b95a62": "relis la cellule 3 : elle affiche x + 1", "a77b3ed193a23dbbff0a1e26a855f366e7d9c0939abd011378dc1763a26a9274": "compte les exécutions de la cellule 2 dans la suite 1, 2, 3, 2, 3", "bd2f1ef5e7c54a0d32ef18b594977477a0a049b9e3a47d97e2d0733b28bb569d": "le noyau garde l'ordre où tu as exécuté les cellules, pas celui où elles sont écrites : refais la suite 1, 2, 3, 2, 3"}, "sign": 1}
📝 Ex B7.Q1a : réponse enregistrée (int).
WB_ANSWER {"hash": "f0285ae738d3da1a2fbf9ad60d0577d9d68870fb765ab2cda5304442cf59b85d", "id": "B7.Q1b", "kind": "str", "mistakes": {"1ec5ee77a9be8d1ab65a86b22ae9c25d62f3da1345f83b095c697d1eea81c4c1": "Jupyter n'exécute que les cellules du fichier : relis « Ce que cache un notebook » (§207.1)", "5573f62f405a12424a023bd031c0966578c56f98a6e021221f1e68af77d8b4ae": "

**B7.Q2 — src/, tests/, configs/, scripts/ : où va quoi ?**

In [4]:
wb.record("B7.Q2a", "A", mistakes={"un fichier de tests vérifie le code ; l'API, elle, importe cette fonction": "B", "configs/ ne contient que des réglages, pas du code": "C", "scripts/ range ce qu'on lance à la main ; cette fonction est importée par l'entraînement et l'API": "D"})
wb.record("B7.Q2b", "D", mistakes={"le package ne contient que le code qu'importent l'entraînement et le service : ce script, on le lance": "A", "ce script ne vérifie pas le code, il prépare les données": "B", "configs/ ne contient que des réglages, pas du code": "C"})
wb.record("B7.Q2c", "C", mistakes={"un fichier YAML de réglages n'est pas du code importable": "A", "ce fichier décrit un entraînement ; il ne teste rien lui-même": "B", "scripts/ range du code à lancer ; un fichier YAML est un réglage": "D"})
wb.record("B7.Q2d", "B", mistakes={"le package ne contient que le code qui sert en production": "A", "configs/ ne contient que des réglages": "C", "où pytest cherche-t-il les tests d'un dépôt ? Relis l'arborescence de 207.1": "D"})
wb.record("B7.Q2e", False, mistakes={"relis le paragraphe qui suit l'arborescence (§207.1) : les données et les sorties des runs vont-elles dans git ?": True})
wb.record("B7.Q2f", "B", mistakes={"relis l'avantage du layout src/ (§207.1) : le dossier courant suffit-il pour importer le package ?": "A", "pytest installe-t-il quoi que ce soit ? Relis le layout src/ (§207.1)": "C"})
wb.record("B7.Q2g", True, mistakes={"relis [project.scripts] (§207.1) : que crée cette ligne à l'installation ?": False})

WB_ANSWER {"hash": "9d95322120c26b2afecc429aee86d7dfb20ed23162239e00beb4e291a0f3c6ad", "id": "B7.Q2a", "kind": "str", "mistakes": {"6946269fef036f59ae3dcc288c370126f6e301c3910a40d3fb47cbafe21fc46a": "un fichier de tests vérifie le code ; l'API, elle, importe cette fonction", "c2a337d958c229641b498eb4d4f5bdd0bcab5766e69620615c671d2eaa710228": "scripts/ range ce qu'on lance à la main ; cette fonction est importée par l'entraînement et l'API", "f4fe70d4259a06e0559463b0ff67b84c8e98fa5679b1f5cbf290622dfa08be21": "configs/ ne contient que des réglages, pas du code"}}
📝 Ex B7.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "514e71f81aa6170e1155b0c7d2842489151d6a6bf6a9d9c9d78561625bf41552", "id": "B7.Q2b", "kind": "str", "mistakes": {"71c71ac893b0e46a8ecf90f3fb62abb5a7d78425ade56d35208c7a61065cea72": "ce script ne vérifie pas le code, il prépare les données", "c48c5cd96af9c5e35ec0b9fa010485a39cf273ce87b51ef7a40ed578081c9531": "le package ne contient que le code qu'importent l'entraînement 

**B7.Q3 — print ou logging : niveaux, destinations et messages utiles**

In [5]:
wb.record("B7.Q3a", "B", mistakes={"DEBUG sert au détail pour chercher un bug : une fin d'epoch en est-elle un ?": "A", "WARNING signale l'inattendu : une fin d'epoch l'est-elle ?": "C", "ERROR dit qu'une opération a échoué : relis les niveaux (§207.2)": "D"})
wb.record("B7.Q3b", "C", mistakes={"un réglage absent, remplacé par une valeur par défaut : un simple détail ?": "A", "est-ce une étape normale, ou quelque chose d'inattendu qui n'empêche pas de continuer ?": "B", "le programme continue : une opération a-t-elle échoué ?": "D"})
wb.record("B7.Q3c", Q3C, mistakes={"un niveau WARNING laisse-t-il passer les messages INFO ? Relis les niveaux (§207.2)": 5,
          "un message ERROR est plus grave qu'un WARNING : passe-t-il le niveau choisi ?": 1,
          "un niveau WARNING laisse-t-il passer DEBUG et INFO ?": 6})
wb.record("B7.Q3d", False, mistakes={"relis « Le logging plutôt que print » (§207.2) : où configure-t-on les handlers, et combien de fois ?": True})
wb.record("B7.Q3e", "C", mistakes={"quel fait vérifiable ce message donne-t-il ? Relis la fin de « Le logging plutôt que print »": "A", "un message utile donne des valeurs : lequel en donne ?": "B"})
wb.record("B7.Q3f", 0, mistakes={"le except affiche l'erreur, puis le script va au bout : comment finit-il ? Relis « Les codes de sortie » (§207.2)": 1,
          "2 est le code d'argparse devant une mauvaise option : relis « Les codes de sortie »": 2})

WB_ANSWER {"hash": "0bed5eeb746dc46f5044971c9c82cc3839eb68b565284af89486f9928db10f51", "id": "B7.Q3a", "kind": "str", "mistakes": {"789ae652c581b62fa26ccb3cf0e3c07491b9d4caeb7cc1795703636d58136da4": "DEBUG sert au détail pour chercher un bug : une fin d'epoch en est-elle un ?", "84066a21f5ba6b074d6966c8713fc9d534f81766e25be343a986447cb263022a": "ERROR dit qu'une opération a échoué : relis les niveaux (§207.2)", "bf34158db617480ac67bea97c5e3546ec65cf1f3bc0e2a36285890fd09aabe7f": "WARNING signale l'inattendu : une fin d'epoch l'est-elle ?"}}
📝 Ex B7.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "289569a0791c35ab21c0a9dc7ae10c60c244ce87412985c1f9e044267d36571a", "id": "B7.Q3b", "kind": "str", "mistakes": {"4d9704d3e7e14f4c62f15ce7d7f0601a3cf8001686806b15a9e245672fda1f4f": "est-ce une étape normale, ou quelque chose d'inattendu qui n'empêche pas de continuer ?", "7ffc12c4beb2f76368b6c2acaf80e4617884fd6f93d9e33a21a089da9b1aa598": "un réglage absent, remplacé par une valeur par défaut 

**B7.Q4 — Ce qui va dans la configuration, et ce qui ne doit jamais y aller**

In [6]:
wb.record("B7.Q4a", "ACD", mistakes={"il manque au moins une lettre : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'D', 'C', 'AC', 'AD', 'CD', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'ABC', 'BD', 'AB', 'B', 'BCD', 'BC', 'ABD'}})
wb.record("B7.Q4b", "A", mistakes={"relis le rappel outil sur PyYAML (§207.3) : que faut-il pour qu'un nombre en notation scientifique soit lu comme un float ?": "B", "des guillemets donnent-ils un nombre ou une chaîne en YAML ?": "C"})
wb.record("B7.Q4c", 2, mistakes={"relis main de la fiche (§207.2) : que rend-il quand load_config lève une ValueError ?": 0,
          "1 est le code d'une erreur générale : relis le except de main (§207.2)": 1})
wb.record("B7.Q4d", True, mistakes={"relis « Variables d'environnement et secrets » (§207.9) : que garde une image de chacune de ses couches ?": False})
wb.record("B7.Q4e", "C", mistakes={"un commit suivant efface-t-il le précédent de l'historique ?": "A", "un dépôt privé se clone, se partage et peut devenir public : relis 207.9": "B"})
wb.record("B7.Q4f", False, mistakes={"relis le commentaire de __post_init__ dans TrainConfig (§207.3) : une dataclass vérifie-t-elle les types ?": True})

WB_ANSWER {"hash": "f9e9974f6deb83829f93e8c1c6a7e5cbad64b77222b268d3521250feede0727e", "id": "B7.Q4a", "kind": "str", "mistakes": {"1190f13b051eaaab8197b43aba801b77a6624a80264b61a92616321463fc30d9": "il manque au moins une lettre, et une autre n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)", "397b7fe7a842664f1ff1b78876b3da18ff28bec84a3ba529f9abc72182693cd3": "au moins une des lettres choisies n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)", "48f4a199796cee35ae33d9aa3c18d391f7c9e2ee8c59d70ef3b651bcccd51dc0": "il manque au moins une lettre, et une autre n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)", "4d75aad1bc629775119b42eb39b19f6df248fdefffc87d66e26e491c3e5bcbb6": "il manque au moins une lettre : relis 207.3 et « Variables d'environnement et secrets » (§207.9)", "a13a92ca1adbb165b97d9cad27fcc42ed748c48a026955b2d1fca48297a97879": "il manque au moins une lettre, et une autre n

**B7.Q5 — Même graine, résultats différents : les sources de non-déterminisme**

In [7]:
wb.record("B7.Q5a", "C", mistakes={"l'initialisation de nn.Linear tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?": "A", "le dropout tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?": "B"})
wb.record("B7.Q5b", True, mistakes={"relis le paragraphe qui suit seed_everything (§207.4) : l'addition des flottants est-elle associative ?": False})
wb.record("B7.Q5c", "C", mistakes={"c'est le comportement sans warn_only : relis l'encadré 🕰️ de 207.4": "A", "PyTorch ne remplace pas une opération par une autre : relis l'encadré 🕰️ de 207.4": "B"})
wb.record("B7.Q5d", False, mistakes={"calcule les deux sommes en Python : l'addition des flottants est-elle associative ?": True})
wb.record("B7.Q5e", "B", mistakes={"sans generator, où le DataLoader tire-t-il son ordre ? Relis le paragraphe qui suit seed_everything (§207.4)": "A", "un tirage de plus n'est pas une erreur : que change-t-il au générateur global ?": "C"})

WB_ANSWER {"hash": "69f94eaf2a8f07454de325c9483a70d22f84c496048c50d6d99823cf5d7db2c1", "id": "B7.Q5a", "kind": "str", "mistakes": {"0a6368ccdd96c4ae4d04088493c16d0cd011bc282379c2850f78554e374be681": "l'initialisation de nn.Linear tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?", "ff884821df42b00a71e006c3dc5fae47012810edda2149029e465df147f9d50a": "le dropout tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?"}}
📝 Ex B7.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "2eae29837880b4b31d6d37273204134e0bcd01650bc9b0939a75a089398643eb", "id": "B7.Q5b", "kind": "bool", "mistakes": {"6a16c1c5db501a5947f26c33a99165af9355999272705134bf468fe6b74cc17f": "relis le paragraphe qui suit seed_everything (§207.4) : l'addition des flottants est-elle associative ?"}}
📝 Ex B7.Q5b : réponse enregistrée (bool).
WB_ANSWER {"hash": "b941a1b6bb8becf159c97f62720b7f1a5bdf675abab61fcf8fae643125ae97b8", "id": "B7.Q5c", "kind": "str", "mistakes": {"aa2060dc6e2e77556af1ff

**B7.Q6 — requirements.txt, fichier de verrouillage, pyproject.toml : qui fige quoi ?**

In [8]:
wb.record("B7.Q6a", "C", mistakes={"une borne >= dit-elle une version exacte ? Relis « pyproject.toml et installation éditable » (§207.1)": "A", "Colab installe ses propres versions : que dit une borne inférieure ?": "B"})
wb.record("B7.Q6b", "A", mistakes={"pyproject.toml donne des bornes : relis « Les versions figées et l'empreinte des données » (§207.4)": "B", "un README décrit le projet, il ne fige rien": "C"})
wb.record("B7.Q6c", False, mistakes={"relis « Les versions figées » (§207.4) : une bibliothèque qu'on publie garde-t-elle des versions exactes ou des bornes ?": True})
wb.record("B7.Q6d", "A", mistakes={"le run a tourné avec du code modifié : ce code est-il tout entier dans a1b2c3d ?": "B", "un commit enregistré reste valide : que manque-t-il au run ?": "C"})
wb.record("B7.Q6e", True, mistakes={"un fichier de verrouillage fige-t-il des versions exactes ou des bornes ? Relis « Les versions figées » (§207.4)": False})

WB_ANSWER {"hash": "e9ea2cb58b3f0684a6a7febacbf7b8996866ab4ba2fb55e72a0a5631e18ff2f7", "id": "B7.Q6a", "kind": "str", "mistakes": {"2ca5919aec7b19fec472e868a6c7cd09e35f87a2751da12334900430c6c1b9b6": "une borne >= dit-elle une version exacte ? Relis « pyproject.toml et installation éditable » (§207.1)", "bcb3be021a46ad62a214399535d7e8ada46a2f21f393e44fe7b06c4732813d60": "Colab installe ses propres versions : que dit une borne inférieure ?"}}
📝 Ex B7.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "7925241a34b466d33c22605ef5c86d64f31120d19be855b267cfa1a5a0c89a4f", "id": "B7.Q6b", "kind": "str", "mistakes": {"6eb84f0d8d155c1afd728becac1ae5779aec58be3b515e3621d700d4a0d0d863": "pyproject.toml donne des bornes : relis « Les versions figées et l'empreinte des données » (§207.4)", "92d8498e61a46696b3833392b90e661fbfb9450e2e144fe968af4112450d76bd": "un README décrit le projet, il ne fige rien"}}
📝 Ex B7.Q6b : réponse enregistrée (str).
WB_ANSWER {"hash": "7dcf7f150a5c86925c4e19749e4d61d4b80

**B7.Q7 — Que tester dans un projet de machine learning ?**

In [9]:
wb.record("B7.Q7a", "B", mistakes={"la forme de la sortie dépend-elle de zero_grad ? Relis « Quatre familles de tests » (§207.6)": "A", "le schéma des données ne voit pas la boucle d'entraînement : relis « Quatre familles de tests »": "C"})
wb.record("B7.Q7b", "C", mistakes={"l'entraînement fumée vérifie que tout va au bout, pas le contenu des données": "A", "apprendre 16 images par cœur dit-il quelque chose du recouvrement des jeux ?": "B"})
wb.record("B7.Q7c", "A", mistakes={"relis la puce « bout en bout » (§207.6) : l'entraînement fumée juge-t-il l'accuracy ?": "B", "l'entraînement fumée lance un entraînement, pas l'API": "C"})
wb.record("B7.Q7d", False, mistakes={"relis la puce « bout en bout » (§207.6) : que vérifie l'entraînement fumée, et que ne juge-t-il pas ?": True})
wb.record("B7.Q7e", Q7E, decimals=1, mistakes={"relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par leur moyenne ?": float(np.mean(list(SECTIONS_Q7.values()))),
          "relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par leur total ?": float(sum(SECTIONS_Q7.values())),
          "relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par la plus grande ?": 4.0})

WB_ANSWER {"hash": "9db503750126ed5d1ed6504018ccc50d7becc6eccdee33f24cef1918e3c0ba8c", "id": "B7.Q7a", "kind": "str", "mistakes": {"2167b17c02d0db19852cb2efabe73524687aaf1a944ef592c2470658580971dc": "le schéma des données ne voit pas la boucle d'entraînement : relis « Quatre familles de tests »", "d8b183720a0aaab12cdfb03050b2ecf850ff407c6769a82d1611c48ed647e556": "la forme de la sortie dépend-elle de zero_grad ? Relis « Quatre familles de tests » (§207.6)"}}
📝 Ex B7.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "f5611ec22f0bdf675a92bed7404ecbf942a70ab6c4fa5f80d15beade923a049f", "id": "B7.Q7b", "kind": "str", "mistakes": {"183a1dd759cf3510a9e6e0818d5ef47b7c5f8f47f06fb6075bc17ce9d3e74382": "apprendre 16 images par cœur dit-il quelque chose du recouvrement des jeux ?", "685954d15fe389783fa98668d3f1071f4ab18b5fa7a87ef63df27662ef7b7a64": "l'entraînement fumée vérifie que tout va au bout, pas le contenu des données"}}
📝 Ex B7.Q7b : réponse enregistrée (str).
WB_ANSWER {"hash": "c551451

**B7.Q8 — Artefact de modèle : les poids seuls ne suffisent pas**

In [10]:
wb.record("B7.Q8a", "ABD", mistakes={"il manque au moins une lettre : relis « L'artefact du modèle » (§207.7)": {'D', 'BD', 'AB', 'B', 'AD', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis « L'artefact du modèle » (§207.7)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)": {'ABC', 'C', 'AC', 'BCD', 'CD', 'BC', 'ACD'}})
wb.record("B7.Q8b", "A", mistakes={"relis la question : la liste du service n'est pas celle de l'entraînement": "B", "l'indice 3 existe dans toute liste de dix noms : laquelle le service lit-il ?": "C"})
wb.record("B7.Q8c", "C", mistakes={"relis le paragraphe qui suit le code de « L'artefact du modèle » : quels types prennent les métadonnées de safetensors ?": "A", "safetensors ignore-t-il ce qu'il ne sait pas écrire ? Relis « L'artefact du modèle » (§207.7)": "B"})
wb.record("B7.Q8d", True, mistakes={"relis le training-serving skew (§207.7) : des statistiques recalculées sur la production suivent-elles celles de l'entraînement ?": False})
wb.record("B7.Q8e", "B", mistakes={"le dropout agit en mode entraînement : que fait model.eval() ?": "A", "un modèle en mode entraînement calcule-t-il quand même ? Relis la fin de « L'artefact du modèle »": "C"})

WB_ANSWER {"hash": "99f2424911a679fad6bf3da6c6a2754ea9a4ee950fb3187563fec0f6e20369b4", "id": "B7.Q8a", "kind": "str", "mistakes": {"10bf94363465dde107e142525b74a4a8065fdd3dabb992c64dd09ee6ec1c481f": "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)", "21383226ff38b0574049a6842ce9287390682a1af27678504944dff477601b4c": "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)", "2a2406424f70218fc1ae2175dbf07fea8c99a181e9239171ca1165b247f2efcf": "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)", "2c6940f3f2a10a82157c7c84e37d89d5cea289af37fab1ac5cb2ee554e935f0a": "il manque au moins une lettre : relis « L'artefact du modèle » (§207.7)", "4e584ee318c6571d27cc42511d440c3f1333f6921a58dab6498e115241b37d0c": "il manque au moins une lettre : relis « L'artefact du modèle » (§207.7)", "4f850be12687321afb7b60317fb1fbf12ab978e0d09fc4d047851839ca52

**B7.Q9 — Latence p50, p95 et débit : ce que ressent l'utilisateur**

In [11]:
wb.record("B7.Q9a", round(Q9A), mistakes={"relis la question : la moyenne pèse chaque requête, et 900 requêtes prennent 10 ms": 10,
          "c'est la moyenne des deux durées, sans tenir compte des effectifs": 105,
          "relis la question : on demande la moyenne": 200})
wb.record("B7.Q9b", round(Q9B), mistakes={"relis la question : on demande le p95, pas la moyenne": 29,
          "compte les requêtes de 10 ms : laissent-elles 95 % des mesures en dessous de cette valeur ?": 10,
          "c'est la moyenne des deux durées : le p95 se lit dans les mesures triées": 105})
wb.record("B7.Q9c", True, mistakes={"imagine 1 000 requêtes de 29 ms chacune : compare leur p95 à celui de l'énoncé, à moyenne égale": False})
wb.record("B7.Q9d", round(Q9D), mistakes={"convertis 150 ms en secondes avant de multiplier": 9000,
          "relis la loi de Little (§207.7) : L multiplie le débit par le temps de réponse": 400})
wb.record("B7.Q9e", round(Q9E), mistakes={"le débit divise la taille du batch par sa durée": 26})
wb.record("B7.Q9f", "A", mistakes={"une image d'un batch attend-elle les autres ? Relis la fin de « Latence, percentiles et débit » (§207.7)": "B", "relis la fin de « Latence, percentiles et débit » : que gagne-t-on à grouper, même sur CPU ?": "C"})

WB_ANSWER {"hash": "0dd71023e8dc038ad784ceaec27d2ad8cd6a0b61d1bd6a45fd08fd4c65593ddb", "id": "B7.Q9a", "kind": "int", "magnitude": 1, "mistakes": {"18a69b7b9d26f79e20cd2c24234aa89ca84f567ad47ba79333a0b9d86c8d1c79": "relis la question : on demande la moyenne", "6bb9163a756e598f8b911a9686716653dc6582c1a48223c3031b4ff1655902b2": "c'est la moyenne des deux durées, sans tenir compte des effectifs", "d84e1f0a8e335a451d675157bee3f0d6d9c2c6cae26bb2f456b5445c2ea9a49e": "relis la question : la moyenne pèse chaque requête, et 900 requêtes prennent 10 ms"}, "sign": 1}
📝 Ex B7.Q9a : réponse enregistrée (int).
WB_ANSWER {"hash": "1f73af8539eae8574cac043277750490ade34fdac30462c60526237df69736be", "id": "B7.Q9b", "kind": "int", "magnitude": 2, "mistakes": {"6c40e8935b935b41564f00cef9a5694e60dc0dccd2a7f6e4f64da9ba4b316f5f": "relis la question : on demande le p95, pas la moyenne", "93e568cd58458e94668ae7c60fe2e7d005380ec71e27c1a0c17721216ec4cf90": "compte les requêtes de 10 ms : laissent-elles 95 % des 

**B7.Q10 — torch.export, ONNX, TorchScript : lequel choisir aujourd'hui ?**

In [12]:
wb.record("B7.Q10a", "B", mistakes={"relis l'encadré 🕰️ de 207.8 : que dit PyTorch 2.11 de torch.jit.script ?": "A", "un objet sauvé par pickle se recharge-t-il sans le code de sa classe ?": "C"})
wb.record("B7.Q10b", "A", mistakes={"torch.compile accélère un modèle dans un programme Python : produit-il un fichier ?": "B", "un state_dict se recharge-t-il sans PyTorch ni le code du modèle ?": "C"})
wb.record("B7.Q10c", False, mistakes={"relis l'encadré 🕰️ de 207.8 : que produit torch.compile ?": True})
wb.record("B7.Q10d", "B", mistakes={"relis le paragraphe sur dynamic_shapes (§207.8) : que devient la taille de batch d'un export sans dynamic_shapes ?": "A", "un modèle exporté complète-t-il une entrée ? Relis le paragraphe sur les gardes (§207.8)": "C"})
wb.record("B7.Q10e", True, mistakes={"relis l'encadré 🕰️ de 207.8 : sur quoi s'appuie torch.onnx.export depuis la 2.9 ?": False})
wb.record("B7.Q10f", "A", mistakes={"un graphe figé peut-il contenir un if Python sur une valeur des données ? Relis le début de « torch.export et ONNX »": "B", "garder la seule branche de l'exemple, c'était le défaut de torch.jit.trace : relis le début de « torch.export et ONNX »": "C"})

WB_ANSWER {"hash": "99092009607c7cbf013311268473b981883bb16df141eb2fd6fff969c40580b3", "id": "B7.Q10a", "kind": "str", "mistakes": {"0c817e87325f61bb30963ef0ceb8f1b0280c7c8d5f66995ae5a60d8f19203b56": "relis l'encadré 🕰️ de 207.8 : que dit PyTorch 2.11 de torch.jit.script ?", "37353021c62937d9c498f88473575735beda079bcd3022ead92ce074cfb7bfb1": "un objet sauvé par pickle se recharge-t-il sans le code de sa classe ?"}}
📝 Ex B7.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "41941f2b6a55fb63562ff6a709ebd8ed0b142e86a33fb41dd6ba5a2d6d366cb6", "id": "B7.Q10b", "kind": "str", "mistakes": {"2627dfde651dfbc9df7e63092e4762b27e3ac7d275ec0cbb72d5f4007e5fda07": "torch.compile accélère un modèle dans un programme Python : produit-il un fichier ?", "5413ea0efefb3ac2529430db445753ffc7d3c7abfae4c37e269e2a2a68533aef": "un state_dict se recharge-t-il sans PyTorch ni le code du modèle ?"}}
📝 Ex B7.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "2da863ba30e214cc768bcdf241400f5b2fefb87c62f3d461070

**B7.Q11 — Écart entraînement-production et dérive des données : vrai ou faux**

In [13]:
wb.record("B7.Q11a", True, mistakes={"relis « Pourquoi un modèle se dégrade sans changer » (§207.10)": False})
wb.record("B7.Q11b", False, mistakes={"relis la définition du training-serving skew (§207.7) : d'où vient l'écart ?": True})
wb.record("B7.Q11c", True, mistakes={"relis la fin de « Le test de Kolmogorov-Smirnov » (§207.10) : que devient une p-valeur avec beaucoup de données ?": False})
wb.record("B7.Q11d", True, mistakes={"relis ce qu'on surveille sur les sorties (§207.10) : faut-il attendre les labels ?": False})
wb.record("B7.Q11e", "B", mistakes={"le monde a-t-il changé, ou les deux chaînes de préparation diffèrent-elles ?": "A", "la relation entre une image et sa classe a-t-elle changé ? Relis 207.7 et 207.10": "C"})
wb.record("B7.Q11f", False, mistakes={"relis la fin de « Les seuils » (§207.10) : quand fixe-t-on les bornes des intervalles ?": True})

WB_ANSWER {"hash": "34dbc1e843c450945954cb95ef24b1b43eab99929b2a8e52f011ce84c47dc6ea", "id": "B7.Q11a", "kind": "bool", "mistakes": {"9b84061b9ad9aac150bcd853c9b08bb83ab930e56aafe00d77ad0aac1e45c8c7": "relis « Pourquoi un modèle se dégrade sans changer » (§207.10)"}}
📝 Ex B7.Q11a : réponse enregistrée (bool).
WB_ANSWER {"hash": "ea5e31724ef8454d6df706f232b02ce28c63215667d16aea97cecf7281c7cb50", "id": "B7.Q11b", "kind": "bool", "mistakes": {"36fefe0014597072aa5ee6ad3d7a70a316ba673c2a42be37d61e01f5623e96a3": "relis la définition du training-serving skew (§207.7) : d'où vient l'écart ?"}}
📝 Ex B7.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "68314b862542832cd6911467ac382baff7afe8599f90966e0906f1881ce5be1f", "id": "B7.Q11c", "kind": "bool", "mistakes": {"fe6e326e241ad4bd4a78df77e38da5bf37025ac7105429c5a6948fd276274cd1": "relis la fin de « Le test de Kolmogorov-Smirnov » (§207.10) : que devient une p-valeur avec beaucoup de données ?"}}
📝 Ex B7.Q11c : réponse enregistrée (bool).
WB

**B7.R1 — Ce qu'une ligne de journal d'expériences doit contenir**

In [14]:
wb.record("B7.R1a", "ABC", mistakes={"il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.5": {'C', 'BC', 'AB', 'AC', 'B', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.5": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.5": {'D', 'BD', 'AD', 'BCD', 'CD', 'ABD', 'ACD'}})
wb.record("B7.R1b", "A", mistakes={"relis la fin du paragraphe qui suit log_run (§207.5) : que fausse l'effacement d'un run raté ?": "B", "remplacer la ligne, c'est réécrire le passé : relis la docstring de log_run (§207.5)": "C"})
wb.record("B7.R1c", True, mistakes={"relis le paragraphe qui suit log_run (§207.5) : un champ ajouté gêne-t-il les anciennes lignes ?": False})
wb.record("B7.R1d", R1D, mistakes={"un journal JSONL s'écrit en ajoutant des lignes : relance-t-on un run en réécrivant la sienne ?": 7,
          "le journal garde-t-il les sept premiers runs ? Relis la docstring de log_run (§207.5)": 2})
wb.record("B7.R1e", "C", mistakes={"relis new_run_id (§207.5) : de quoi l'identifiant est-il fait ?": "A", "relis new_run_id (§207.5) : le commit y figure-t-il ?": "B"})

WB_ANSWER {"hash": "6e252cab2f1fcecf4b6fa2bc44150c3d6fb06ffb24b89345d395af7b2d42d953", "id": "B7.R1a", "kind": "str", "mistakes": {"04b6b649e63533014a31fc2c68cb38ad36d1d86aa1dc6350a44ca816a99418f0": "il manque au moins une lettre, et une autre n'a pas sa place : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.5", "0a1e87f093f4917d576014627341f9f1855b8cf4d820b239df38223e2f601944": "il manque au moins une lettre, et une autre n'a pas sa place : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.5", "256a8a0b2d41fed3d4e8752c0d178ecfb51e20f44bbc5a7280e0ec93ff941d4b": "il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.5", "25c969bedafe23a73677efcfa6de13ed1a3f7c79af15b259adef6bc3ec2a01de": "il manque au moins une lettre, et une autre n'a pas sa place : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.5", "568ce87fd3e84626681f24bd146928c24e22c6c42804c7967e9a25be9a8efc83": "il manque au moins une lettre, et une autre n'a pas

**B7.R2 — Checkpoint complet, state_dict et weights_only**

In [15]:
wb.record("B7.R2a", "ABD", mistakes={"il manque au moins une lettre : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'D', 'BD', 'AB', 'B', 'AD', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'ABC', 'C', 'AC', 'BCD', 'CD', 'BC', 'ACD'}})
wb.record("B7.R2b", "B", mistakes={"l'optimiseur sert-il à prédire ? Relis §23.9.2 du ch. 23 (« Les poids seuls »)": "A", "l'état de l'optimiseur contient-il les poids ? Relis §23.9.2 du ch. 23": "C"})
wb.record("B7.R2c", True, mistakes={"relis l'encadré 🕰️ du ch. 23 sur les formats de sauvegarde : que refuse weights_only=True ?": False})
wb.record("B7.R2d", False, mistakes={"relis §23.9.2 du ch. 23 : que faut-il faire avant load_state_dict ?": True})
wb.record("B7.R2e", "A", mistakes={"relis §23.9.1 du ch. 23 : que perd-on sans l'état de l'optimiseur ?": "B", "le load_state_dict du modèle se soucie-t-il de l'optimiseur ?": "C"})

WB_ANSWER {"hash": "a5f60fdeb65c38e980086d9bd119ed6fdd0e806566e67c3dd9604c42256b3a50", "id": "B7.R2a", "kind": "str", "mistakes": {"4a10d7a1a34860f1835d57638ae5fd4d11c8a3c10203a321412a5ea55dbd62e2": "il manque au moins une lettre, et une autre n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "565861efc990feef588b457f6ec779287bf09a5edad5582c8a5c5f612d0fa8e9": "il manque au moins une lettre, et une autre n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "5aebe8013527d03c7b0f03d3bb84c7869cf1e274b1e882216b911717e998be4f": "au moins une des lettres choisies n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "5b7c3c7ceaa4d7d093b9700526a870ec7fb98dd0c81dc97eccb287eac05e9e9c": "il manque au moins une lettre : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "5bd75c32ed760ae23b4e02b0687442c4f2e015a817bc826fd5bb4a674ef0152c": "il manque au moins une lettre : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "6e4ae3dc7560313

**B7.R3 — Sauvegarder un Pipeline et figer les versions**

In [16]:
wb.record("B7.R3a", "A", mistakes={"relis la première règle de « sauvegarder un modèle et ses versions » (ch. 15)": "B", "des coefficients sans leur prétraitement servent-ils à prédire ? Relis la première règle du ch. 15": "C"})
wb.record("B7.R3b", False, mistakes={"relis la deuxième règle du ch. 15 : que garantit la documentation d'une version à l'autre ?": True})
wb.record("B7.R3c", "B", mistakes={"un FutureWarning annonce un changement d'API : relis la deuxième règle du ch. 15": "A", "relis la deuxième règle du ch. 15 : scikit-learn prévient-il, depuis la 1.3 ?": "C"})
wb.record("B7.R3d", True, mistakes={"relis la troisième règle du ch. 15 : que fait le chargement d'un pickle ?": False})
wb.record("B7.R3e", "ABC", mistakes={"il manque au moins une lettre : relis la deuxième règle du ch. 15 et « Variables d'environnement et secrets » (§207.9)": {'C', 'BC', 'AB', 'AC', 'B', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis la deuxième règle du ch. 15 et « Variables d'environnement et secrets » (§207.9)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis la deuxième règle du ch. 15 et « Variables d'environnement et secrets » (§207.9)": {'D', 'BD', 'AD', 'BCD', 'CD', 'ABD', 'ACD'}})
wb.record("B7.R3f", "C", mistakes={"skops.io vérifie les types avant de charger, mais en Python : relis la troisième règle du ch. 15": "A", "joblib demande Python et scikit-learn : relis la troisième règle du ch. 15": "B"})

WB_ANSWER {"hash": "e38281ba172b9457fed188dc5f94428d489b0e793d4ff31170d937b2a3f51161", "id": "B7.R3a", "kind": "str", "mistakes": {"0f9e6251328fd298269ea7ef86129e2822e7d8dec3285b3a401623dd2c5c73ac": "des coefficients sans leur prétraitement servent-ils à prédire ? Relis la première règle du ch. 15", "3e1bf7d29404a62d2031750d3ab9b59a1852071b7aef5ad67b3fb547e15390da": "relis la première règle de « sauvegarder un modèle et ses versions » (ch. 15)"}}
📝 Ex B7.R3a : réponse enregistrée (str).
WB_ANSWER {"hash": "528b4db8d3df07989768a2feaed69a9d1cee63a743bb5b0f94f8f3f52b23c1db", "id": "B7.R3b", "kind": "bool", "mistakes": {"8272eb99b515ec0ff2fa7895cdc2839991fe2c5dbb66888634d32bb0517751d9": "relis la deuxième règle du ch. 15 : que garantit la documentation d'une version à l'autre ?"}}
📝 Ex B7.R3b : réponse enregistrée (bool).
WB_ANSWER {"hash": "8c852ed41f363b3236da4f7bc8fb98dfda8e06c773a31f2c93378bcaa70c5fa8", "id": "B7.R3c", "kind": "str", "mistakes": {"82fdc42676b0289f2db40cb1318c20218d8e64

**Ex B7.1 — Taille d'un artefact : float32, bfloat16, int8, et la mémoire au chargement**

In [17]:
wb.record("B7.1a", F32_B71, decimals=1, mistakes={"un float32 occupe 4 octets : tu as compté 4 bits": N_B71 * 4 / 8 / MB,
          "relis la convention de l'énoncé : 1 Mo = 10⁶ octets": F32_MIB_B71,
          "un float32 occupe 4 octets par paramètre, pas 1": N_B71 / MB,
          "tu as compté en bits : un octet en fait 8": 32 * N_B71 / MB})
wb.record("B7.1b", F32_MIB_B71, decimals=1, mistakes={"relis la question : en Mio, avec 1 Mio = 2²⁰ octets": F32_B71,
          "1 Mio = 2²⁰ = 1 048 576 octets : relis la convention de l'énoncé": 4 * N_B71 / 1_024_000})
wb.record("B7.1c", BF16_B71, decimals=1, mistakes={"un bfloat16 occupe 2 octets par paramètre, pas 4": F32_B71,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": 2 * N_B71 / MIB,
          "le bfloat16 a 16 bits : combien d'octets ?": N_B71 / MB})
wb.record("B7.1d", INT8_B71, decimals=1, mistakes={"un int8 occupe 1 octet, pas 2": BF16_B71,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": N_B71 / MIB})
wb.record("B7.1e", DYNQ_B71, decimals=1, mistakes={"seules les matrices des couches denses passent en int8 : relis l'énoncé": INT8_B71,
          "les biais des couches denses restent en float32 : relis l'énoncé": (LIN_W_B71 + LIN_B_B71 + 4 * (N_B71 - LIN_W_B71 - LIN_B_B71)) / MB,
          "n'oublie pas les paramètres hors des matrices denses, en float32": LIN_W_B71 / MB,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": DYNQ_B71 * MB / MIB})
wb.record("B7.1f", RATIO_B71, decimals=1, mistakes={"relis la question : e) ne passe pas tous les paramètres en int8": 4.0,
          "vérifie le sens du rapport : a) divisé par e)": 1 / RATIO_B71})
wb.record("B7.1g", PEAK_B71, decimals=1, mistakes={"relis l'énoncé : combien de copies des poids sont en mémoire au moment de load_state_dict ?": F32_MIB_B71,
          "relis l'unité : en Mio": 2 * F32_B71,
          "relis l'énoncé : deux copies des poids, pas trois": 3 * F32_MIB_B71})
wb.record("B7.1h", VGG_BYTES_B71, mistakes={"le state_dict contient aussi les buffers des batchnorms": 4 * VGG_PARAMS_B71,
          "relis l'énoncé : num_batches_tracked est un int64, de 8 octets": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71 + 6),
          "n'oublie pas les six compteurs num_batches_tracked": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71),
          "relis l'énoncé : chaque batchnorm a deux buffers d'une valeur par canal": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71 // 2) + 48})
wb.record("B7.1i", HEADER_B71, mistakes={"les buffers des batchnorms sont aussi dans le fichier : retire-les avec les paramètres": VGG_FILE_B71 - 4 * VGG_PARAMS_B71,
          "relis la question : on demande l'en-tête, ce qui reste une fois les tenseurs retirés": VGG_BYTES_B71})
wb.record("B7.1j", LIN_SHARE_B71, decimals=1, mistakes={"relis la question : en pourcentage": VGG_LINEAR_B71 / VGG_PARAMS_B71,
          "vérifie le sens du rapport : les paramètres de la couche dense divisés par tous les paramètres": 100 * VGG_PARAMS_B71 / VGG_LINEAR_B71})

WB_ANSWER {"decimals": 1, "hash": "bd55758152b071535a6add5e4da375c5e1cd966047acae51ceaaaf0ca54daafc", "hash_coarse": "9f5a187cf3a5cf66e9e1a6ecacb3674170119069f5c1bad99442d9e62dd227dc", "id": "B7.1a", "kind": "float", "magnitude": 2, "mistakes": {"74918b48994066c44b0c8bd261b94822cf64793db9c546910093ae7341fbca62": "un float32 occupe 4 octets par paramètre, pas 1", "7af881a605bcf66d5e451082e892c8ab3240d1152db4032f3c33a33456ce9ea7": "un float32 occupe 4 octets : tu as compté 4 bits", "8a98ca19b376d1f19d30bee68b1fee06d22ba8a5064a4013206ca049e1ee174a": "relis la convention de l'énoncé : 1 Mo = 10⁶ octets", "b4d9bdadaefda287bcf83f52ace89db71ebe3ddb45dbc15d0e62f94e4f5641e8": "tu as compté en bits : un octet en fait 8"}, "sign": 1}
📝 Ex B7.1a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "0d02495aa15139491441f0c41b1b820cef8cefb153ba7ec380c29605a1413f19", "hash_coarse": "d762b958c461860ffda094644e38f30da440f7dd863554548a6d7ff0aae65719", "id": "B7.1b", "kind": "f

**Ex B7.2 — Cinq graines, un intervalle : la dispersion d'une accuracy rapportée**

In [18]:
wb.record("B7.2a", MEAN_A_B72, decimals=1, mistakes={"divise la somme par le nombre de graines": float(A_B72.sum() / 4)})
wb.record("B7.2b", S_A_B72, decimals=3, mistakes={"l'écart-type corrigé divise par n − 1 : relis le rappel maths de 207.4": float(np.std(A_B72)),
          "c'est la variance : prends sa racine": float(np.var(A_B72, ddof=1)),
          "c'est la somme des carrés des écarts : divise-la, puis prends la racine": float(np.sum((A_B72 - A_B72.mean()) ** 2))})
wb.record("B7.2c", SE_A_B72, decimals=3, mistakes={"l'erreur type divise s par √n, avec n = 5 graines": S_A_B72 / 2,
          "l'erreur type divise s par √n, pas par n": S_A_B72 / 5,
          "pars de l'écart-type corrigé, divisé par n − 1 : relis b)": interval_b72(A_B72, ddof=0)[2]})
wb.record("B7.2d", HW_A_B72, decimals=2, mistakes={"avec cinq valeurs, on prend le quantile de Student : relis le rappel maths de 207.4": 1.96 * SE_A_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": 2.571 * SE_A_B72,
          "la demi-largeur multiplie t par l'erreur type, pas par l'écart-type": T4_B72 * S_A_B72,
          "pars de l'écart-type corrigé, divisé par n − 1": interval_b72(A_B72, ddof=0)[3]})
wb.record("B7.2e", LOW_A_B72, decimals=2, mistakes={"la borne basse retire la demi-largeur à la moyenne": MEAN_A_B72 + HW_A_B72,
          "avec cinq valeurs, on prend le quantile de Student : relis le rappel maths de 207.4": MEAN_A_B72 - 1.96 * SE_A_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": MEAN_A_B72 - 2.571 * SE_A_B72})
wb.record("B7.2f", OVERLAP_B72, mistakes={"calcule l'intervalle de B comme celui de A, puis compare leurs bornes": not OVERLAP_B72})
wb.record("B7.2g", MEAN_D_B72, decimals=1, mistakes={"vérifie le sens de la différence : B − A": -MEAN_D_B72})
wb.record("B7.2h", HW_D_B72, decimals=2, mistakes={"avec cinq différences, on prend le quantile de Student : relis le rappel maths de 207.4": 1.96 * SE_D_B72,
          "la demi-largeur multiplie t par l'erreur type des différences, pas par leur écart-type": T4_B72 * S_D_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": 2.571 * SE_D_B72})
wb.record("B7.2i", ZERO_IN_D_B72, mistakes={"calcule les deux bornes de l'intervalle de la différence avec g) et h)": not ZERO_IN_D_B72})

WB_ANSWER {"decimals": 1, "hash": "3cd85131e6de728cc20a2d23a392f66869d21f8d30b1bef06bbc8b2c0b0bc550", "hash_coarse": "4329d37d5acc95ccd6adc68516a9f83869448cd839b994ab6c6b35899094aa99", "id": "B7.2a", "kind": "float", "magnitude": 1, "mistakes": {"73d94fd51d99f72b51739c9a00163272113bf2031cd7b4118aa37de4c3c589b7": "divise la somme par le nombre de graines"}, "sign": 1}
📝 Ex B7.2a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "f83f3bd7e477f9fff21ab3b02e3e7126bef006e2ad0c3f202475954fb0e370c6", "hash_coarse": "8f172f65f040affa78c6be31c716299bcbdc6ec36ec237dd287298d15115a61d", "id": "B7.2b", "kind": "float", "magnitude": -1, "mistakes": {"21406df5f29f787e0b1cb6936218d944c270c9c612d6d71315d5f3cf3e475208": "c'est la variance : prends sa racine", "471a000c04b8ed0f9cb071f482288d20011be9cc84bfb9f91139cc4bbb52406b": "c'est la somme des carrés des écarts : divise-la, puis prends la racine", "9b0592b47e0528a46f0817749645b5784c16899caacb83328de9b1a4945ff42c": "l'écar

**Ex B7.3 — Latence et débit : p50 et p95 sur vingt mesures, batch 1 contre batch 32**

In [19]:
wb.record("B7.3a", MEAN_B73, decimals=2, mistakes={"relis la question : la moyenne des vingt mesures": P50_B73,
          "divise la somme par le nombre de mesures, 20": float(LAT_B73.sum() / 19)})
wb.record("B7.3b", P50_B73, decimals=2, mistakes={"c'est la plus petite valeur qui laisse la moitié des mesures en dessous (method=\"inverted_cdf\") : relis la convention de l'énoncé": 20.5,
          "relis la position : h = 0,5 × (n − 1), avec n = 20 mesures": 20.6,
          "relis la question : on demande le p50, la médiane": MEAN_B73})
wb.record("B7.3c", P95_B73, decimals=2, mistakes={"c'est la convention « la plus petite valeur qui laisse 95 % des mesures en dessous » (inverted_cdf) : relis celle de l'énoncé": 35.7,
          "c'est la moyenne des deux plus grandes mesures : relis la position h de l'énoncé": 41.8,
          "c'est la plus grande mesure : relis la position h de l'énoncé": 47.9,
          "c'est une autre convention (position q × (n + 1)) : relis celle de l'énoncé": float(np.percentile(LAT_B73, 95, method="weibull"))})
wb.record("B7.3d", ABOVE_B73, mistakes={"compte les mesures au-dessus de la moyenne, pas de la médiane": 10,
          "relis la question : au-dessus de la moyenne": 20 - ABOVE_B73})
wb.record("B7.3e", round(THR1_B73), mistakes={"relis la question : le débit se calcule ici avec le p50, pas la moyenne": round(1000 / MEAN_B73),
          "relis la question : le débit se calcule ici avec le p50, pas le p95": round(1000 / P95_B73)},
          fractional="le débit se donne ici en images par seconde, arrondi à l'entier")
wb.record("B7.3f", round(THR32_B73), mistakes={"un batch de 32 images : le débit divise sa taille par sa durée": round(1000 / P50_32_B73),
          "relis la question : le débit se calcule avec le p50 du batch, pas son p95": round(32_000 / P95_32_B73)},
          fractional="le débit se donne ici en images par seconde, arrondi à l'entier")
wb.record("B7.3g", RATIO_B73, decimals=1, mistakes={"vérifie le sens du rapport : f) divisé par e)": 1 / RATIO_B73,
          "le débit ne croît pas comme la taille du batch : calcule-le avec les mesures": 32.0})
wb.record("B7.3h", PER_IMAGE_B73, decimals=2, mistakes={"relis la question : le p50 du batch divisé par 32": P50_32_B73,
          "relis la question : on part du p50 du batch, pas de son p95": P95_32_B73 / 32})
wb.record("B7.3i", SLO_B73, mistakes={"chaque image attend la fin de son batch : compare le p95 d'un batch de 32 à l'objectif": not SLO_B73})
wb.record("B7.3j", PAGE_B73, decimals=2, mistakes={"relis la question : au moins une des huit dépasse le p95": 0.95 ** 8,
          "les probabilités ne s'additionnent pas ici : passe par l'événement contraire": 8 * 0.05,
          "relis la question : huit prédictions, pas une seule": 0.05})

WB_ANSWER {"decimals": 2, "hash": "6bac5d3036ddb4e895ab6befe2aaaba3b69d0e7edfe3411dc5cc7390de21f955", "hash_coarse": "a3a9a9b797358ed5352d72ff2c01443af4db05435c5a0404d408c47a379ca9b5", "id": "B7.3a", "kind": "float", "magnitude": 1, "mistakes": {"5bfdc5949835df66c4b7752c8f528390ea41e0f79330ee3672979ea9bbdd2b34": "divise la somme par le nombre de mesures, 20", "7e0b6bb775363e869dde932b9d9786a67a2f192781c310c842ed56f5834a96a7": "relis la question : la moyenne des vingt mesures"}, "sign": 1}
📝 Ex B7.3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "1237d60b7d935b8b084e1b3c8abedcd8bf766a29ec03345b5ed9c5ae61a02965", "hash_coarse": "1a02c4c153eda98ea4f2e225bcba446d3c37eff723b9fa25de4dc2aaec9cc5d2", "id": "B7.3b", "kind": "float", "magnitude": 1, "mistakes": {"1912ca7c11c51714030ef0dceee9b32b3cff30fa96a5d676c0b37441aa9e2c57": "relis la question : on demande le p50, la médiane", "e49f47d7c87572c19a2a3a25e900351aed770ed9ad37693e38b28638b98d5ba5": "relis la posi

**Ex B7.4 — Indice de stabilité de population (PSI) à la main sur quatre intervalles**

In [20]:
wb.record("B7.4a", C1_W1_B74, decimals=4, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[1], log=np.log10)[0]),
          "vérifie le signe : p − q et ln(p/q) ont toujours le même signe": -C1_W1_B74,
          "relis la formule : la contribution multiplie ln(p/q) par p − q": float(np.log(P_B74[1][0] / 0.25))})
wb.record("B7.4b", PSI_W1_B74, decimals=3, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[1], log=np.log10).sum()),
          "il manque une partie du PSI : relis le lien avec la divergence KL (§207.10)": float(np.sum(P_B74[1] * np.log(P_B74[1] / Q_B74)))})
wb.record("B7.4c", PSI_W2_B74, decimals=3, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[2], log=np.log10).sum()),
          "il manque une partie du PSI : relis le lien avec la divergence KL (§207.10)": KL_PQ_W2_B74,
          "relis la question : on demande le PSI, la somme des quatre contributions (p − q) ln(p/q)": KL_QP_W2_B74})
wb.record("B7.4d", "D", mistakes={"calcule la contribution de I₁ en semaine 2 et compare-la aux trois autres": "A", "calcule la contribution de I₂ en semaine 2 et compare-la aux trois autres": "B", "calcule la contribution de I₃ en semaine 2 et compare-la aux trois autres": "C"})
wb.record("B7.4e", KL_QP_W2_B74, decimals=3, mistakes={"relis la question : q en premier, Σ q ln(q/p)": KL_PQ_W2_B74,
          "relis la question : on demande une seule des deux divergences": PSI_W2_B74,
          "la divergence KL utilise le logarithme naturel ln": float(np.sum(Q_B74 * np.log10(Q_B74 / P_B74[2])))})
wb.record("B7.4f", "B", mistakes={"compare le PSI de la semaine 2 au second seuil, 0,25": "A", "compare le PSI de la semaine 1 au premier seuil, 0,1": "C"})
wb.record("B7.4g", True, mistakes={"calcule ln(0 / 0,25) : que vaut-il pour NumPy, et que devient son produit par (0 − 0,25) ?": False})
wb.record("B7.4h", PSI_W3_B74, decimals=2, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[3], eps=1e-4, log=np.log10).sum()),
          "relis l'énoncé : ε vaut 10⁻⁴ ici": PSI_W3_EPS3_B74})
wb.record("B7.4i", PSI_W3_EPS3_B74, decimals=2, mistakes={"relis la question : ε = 10⁻³": PSI_W3_B74,
          "le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[3], eps=1e-3, log=np.log10).sum())})

WB_ANSWER {"decimals": 4, "hash": "0dd5de8ef69669b49252267202a1a40ebf7581e56c5f068ee7135b256550b5ca", "hash_coarse": "0190b71b33be8a0faaab4c265b1b0a10d2fabf066cd0871c90cd21c658eeb8e5", "id": "B7.4a", "kind": "float", "magnitude": -2, "mistakes": {"21bc36827ca4e78ad6601391ab91493d19e5f42b6e97756bd36979a54c8902e3": "relis la formule : la contribution multiplie ln(p/q) par p − q", "5e07c1695b1bc9a5d4d67fe7555b8b028314a9449dc029df127e8be02e2be2a2": "le PSI utilise le logarithme naturel ln", "fc2a3f1845a05d2d16e366b5c36a1b074e40e8ec6a0ea3b82b757a4b66ef2b4a": "vérifie le signe : p − q et ln(p/q) ont toujours le même signe"}, "sign": 1}
📝 Ex B7.4a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "9803e5e4dcc7623a539cfdbff4dbbdc3a6926acd2db40bd90f2770141f93ae91", "hash_coarse": "a8a4ab28c200ce8b5fc1e7251b205cbbe7f5f444671fb5630aed885fae97e118", "id": "B7.4b", "kind": "float", "magnitude": -2, "mistakes": {"2b1f37643b702710ef8644a654253dcce6a91ef32056e16db350d66b4

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire où va chaque fichier d'un projet de ML (code, tests, configuration, données) et pourquoi un notebook ne suffit pas ?
2. Sais-tu rapporter une accuracy sur cinq graines avec son intervalle, et comparer deux modèles graine par graine ?
3. Sais-tu calculer un p50, un p95 et un débit, et dimensionner un service à la louche ?
4. Sais-tu calculer un PSI, dire ce qu'il mesure, et ce que tu ferais d'une alerte ?

**Pour aller plus loin** : l'article de Sculley et coll. (📄 B7.8) et le ML Test Score de Breck et coll., cités dans la fiche. La suite : le bonus B8, l'apprentissage par renforcement moderne.

*Les exercices de code B7.9 à B7.25 (le projet, le service, l'exploitation) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B7` pour obtenir le notebook complet.*